# Graphify Demo — Knowledge Graph over a Python/PySpark Propensity Pipeline

This notebook walks through:
1. Installing Graphify
2. Running it against a codebase (your propensity pipeline repo)
3. Loading `graph.json` into NetworkX for inspection
4. Querying the graph from Python (via CLI subprocess calls)
5. Visualizing the graph and its communities
6. Tracing feature lineage — the core "why this matters" demo moment

> Set `REPO_PATH` below to the path of the codebase you want to analyze.

In [ ]:
REPO_PATH = "./propensity-pipeline"   # <-- change to your repo path

## 1. Install Graphify

Package name is `graphifyy` (double-y), CLI command is `graphify`.

In [ ]:
!pip install graphifyy --quiet

In [ ]:
!graphify --version

## 2. Run Graphify on the codebase

For a pure Python/PySpark codebase this is **free** — code is parsed locally via tree-sitter,
no LLM call, no API key needed. You only need a backend key if the repo also has docs/PDFs/notebooks
you want semantically extracted (e.g. `--backend claude` with `ANTHROPIC_API_KEY` set).

This writes `graphify-out/graph.json`, `graphify-out/GRAPH_REPORT.md`, and `graphify-out/graph.html`
inside `REPO_PATH`.

In [ ]:
!graphify extract {REPO_PATH}

# If your repo also has markdown docs / notebooks you want semantically linked in, use instead:
# import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."
# !graphify extract {REPO_PATH} --backend claude

## 3. Read the auto-generated report

`GRAPH_REPORT.md` gives you god nodes, surprising connections, extracted "why" comments,
and suggested questions — a fast way to orient in an unfamiliar codebase.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

report_path = Path(REPO_PATH) / "graphify-out" / "GRAPH_REPORT.md"
display(Markdown(report_path.read_text()))

## 4. Load `graph.json` into NetworkX

The output is NetworkX's node-link JSON format, so it loads directly.

In [ ]:
import json
import networkx as nx

graph_path = Path(REPO_PATH) / "graphify-out" / "graph.json"
with open(graph_path) as f:
    graph_data = json.load(f)

G = nx.node_link_graph(graph_data)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

### Inspect node and edge structure

In [ ]:
# Peek at a few nodes
for node_id, attrs in list(G.nodes(data=True))[:5]:
    print(node_id, "->", attrs)

In [ ]:
# Peek at a few edges — note the 'relation' and 'confidence' fields
for u, v, attrs in list(G.edges(data=True))[:5]:
    print(f"{u} --[{attrs.get('relation')} | {attrs.get('confidence')}]--> {v}")

### Find the "god nodes" (highest-degree nodes) yourself in Python

`GRAPH_REPORT.md` already lists these, but it's useful to derive them directly for
a live demo — these are almost certainly your feature engineering core, training
entrypoint, and scoring job.

In [ ]:
degree_ranked = sorted(G.degree, key=lambda x: x[1], reverse=True)[:10]
for node_id, degree in degree_ranked:
    label = G.nodes[node_id].get("label", node_id)
    print(f"{degree:>4}  {label}")

### Breakdown by EXTRACTED vs INFERRED edges

This is the audit-trail distinction worth calling out in a demo: EXTRACTED edges came
straight from parsing (imports, function calls, SQL joins) — ground truth. INFERRED
edges are the model's reasoned guesses, each with a confidence score.

In [ ]:
from collections import Counter

confidence_counts = Counter(attrs.get("confidence") for _, _, attrs in G.edges(data=True))
print(confidence_counts)

## 5. Query the graph from Python

`graphify query` / `graphify path` / `graphify explain` are CLI commands, but they're
trivial to call from a notebook with `subprocess` so you can build the queries programmatically
or display the answers inline.

In [ ]:
import subprocess

def graphify_query(question: str, repo_path: str = REPO_PATH) -> str:
    result = subprocess.run(
        ["graphify", "query", question, "--graph", f"{repo_path}/graphify-out/graph.json"],
        capture_output=True, text=True,
    )
    return result.stdout if result.returncode == 0 else result.stderr

print(graphify_query("what feeds into the propensity score?"))

In [ ]:
def graphify_path(start: str, end: str, repo_path: str = REPO_PATH) -> str:
    result = subprocess.run(
        ["graphify", "path", start, end, "--graph", f"{repo_path}/graphify-out/graph.json"],
        capture_output=True, text=True,
    )
    return result.stdout if result.returncode == 0 else result.stderr

# Impact analysis: trace lineage from a raw table to the final model score
print(graphify_path("raw_transactions_table", "final_model_score"))

In [ ]:
def graphify_explain(node_name: str, repo_path: str = REPO_PATH) -> str:
    result = subprocess.run(
        ["graphify", "explain", node_name, "--graph", f"{repo_path}/graphify-out/graph.json"],
        capture_output=True, text=True,
    )
    return result.stdout if result.returncode == 0 else result.stderr

print(graphify_explain("FeatureEngineeringPipeline"))

## 6. Visualize the graph inline

`graph.html` (from the extract step) is already an interactive viewer — open it directly
in a browser. For an inline matplotlib view of a subgraph (e.g. just the top god nodes and
their neighbors), use NetworkX directly:

In [ ]:
import matplotlib.pyplot as plt

# Take the top 15 highest-degree nodes and their immediate neighbors
top_nodes = [n for n, _ in degree_ranked[:15]]
neighborhood = set(top_nodes)
for n in top_nodes:
    neighborhood.update(G.neighbors(n))

subG = G.subgraph(neighborhood)

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(subG, seed=42, k=0.6)
labels = {n: subG.nodes[n].get("label", n)[:20] for n in subG.nodes}

nx.draw_networkx_nodes(subG, pos, node_size=[subG.degree(n) * 40 for n in subG.nodes], node_color="#4C72B0", alpha=0.85)
nx.draw_networkx_edges(subG, pos, alpha=0.3, arrows=True)
nx.draw_networkx_labels(subG, pos, labels=labels, font_size=7)

plt.title("Propensity Pipeline — God Nodes + Neighborhood")
plt.axis("off")
plt.tight_layout()
plt.show()

### Or embed the full interactive `graph.html` directly in the notebook

In [ ]:
from IPython.display import IFrame

IFrame(src=str(Path(REPO_PATH) / "graphify-out" / "graph.html"), width=1000, height=700)

## 7. Community detection — auto-discovered subsystems

Leiden clustering groups the pipeline into subsystems (e.g. feature engineering,
training, scoring) purely from code structure — no embeddings, no vector store.

In [ ]:
!graphify {REPO_PATH} --cluster-only --resolution 1.5

In [ ]:
# Reload the graph — community labels are now attached to nodes
with open(graph_path) as f:
    graph_data = json.load(f)
G = nx.node_link_graph(graph_data)

communities = Counter(attrs.get("community_name", "unlabeled") for _, attrs in G.nodes(data=True))
for name, count in communities.most_common():
    print(f"{count:>4}  {name}")

## 8. Generate the architecture doc (Mermaid call-flow HTML)

Best single artifact to show a non-technical stakeholder.

In [ ]:
!cd {REPO_PATH} && graphify export callflow-html

## Summary — what to show in the demo

| Step | What it proves |
|---|---|
| `graphify extract` | Local, free parsing of the whole pipeline into a graph |
| `GRAPH_REPORT.md` | Instant orientation: god nodes, surprising connections, why-comments |
| `graphify_query(...)` | Ask architecture questions in plain English |
| `graphify_path(...)` | Feature lineage / impact analysis — "what does changing this break" |
| EXTRACTED vs INFERRED | Audit trail — ground truth vs. reasoned guess, with confidence scores |
| Community detection | Auto-discovered subsystems, no embeddings needed |
| `callflow-html` export | Visual artifact for stakeholders |